[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Mechanistic_Interpretability.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Mechanistic Interpretability

Open the model and find the *mechanism*: we train a tiny transformer on a task with a known algorithm (detecting balanced parentheses), then locate where the network computes what — attention maps, linear probes, and the causal test that separates correlation from mechanism: **activation patching**.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb); [Causal Inference](./Causal_Inference.ipynb) supplies the intervention mindset.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# task with a KNOWN algorithm: is a ()-string balanced? ground truth = running-depth check
VOCAB = {"(": 0, ")": 1, "PAD": 2}
L_seq = 16
def make_batch(B):
    xs, ys, depths = [], [], []
    for _ in range(B):
        if rng.random() < 0.5:                                # balanced: random matched string
            s = []
            depth = 0
            for i in range(L_seq):
                if depth == 0 or (rng.random() < 0.5 and depth < L_seq - i - depth):
                    s.append("("); depth += 1
                else:
                    s.append(")"); depth -= 1
            if depth > 0: s[-depth:] = [")"]*depth
        else:                                                  # corrupt one position
            s = ["(", ")"]*(L_seq//2)
            s = list(rng.permutation(s))
        d = np.cumsum([1 if c == "(" else -1 for c in s])
        xs.append([VOCAB[c] for c in s])
        ys.append(int(d[-1] == 0 and d.min() >= 0))
        depths.append(d)
    return torch.tensor(xs), torch.tensor(ys), np.array(depths)

---
### 🕐 Session 1 of 2 — *Probes & Attention Maps* (~40 min)
**Goal:** train the model; find WHERE the running depth lives with linear probes.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (activation patching).

---

## 2. The Model, and the Hypothesis

💡 **Intuition.** Balanced-parentheses has a one-line algorithm: track the running depth, check it never dips below zero and ends at zero. If our transformer learns the task, *something inside it should represent the running depth*. A **linear probe** — a tiny regression from hidden states to the known quantity — tests exactly that, layer by layer and position by position. Finding a probe that works is evidence of a representation; Session 2 tests whether the model actually *uses* it.

In [ ]:

# YOUR CODE HERE


**What just happened.** A two-block transformer with 32-dimensional embeddings reached **100.0% accuracy** on 2,000 held-out strings after 2,500 steps. The task is solved.

**Which immediately raises the question this workshop exists to answer: solved *how*?** Balanced parentheses has a known algorithm — track the running depth, check it never goes negative and ends at zero. **A network at 100% must be computing something sufficient, but nothing so far says it computes that.** The rest of the notebook is the attempt to find out, and having a known correct algorithm is what makes the attempt checkable.

**Before celebrating the number, look at where the two classes come from, because they are generated differently.** Positives are *constructed* balanced strings, built by a loop that maintains depth. Negatives are random **permutations of eight `(` and eight `)`**. Those are different distributions, and a model could in principle separate them on generator artefacts — local run-length statistics, say — without ever computing a depth. **100% accuracy is consistent with the model having learned the algorithm and also with it having learned a shortcut.**

**The check is fifteen lines and worth running.** Build a test set where *both* classes come from the same generator: take constructed balanced strings and flip one character to make the negatives, which is exactly what `matched_pair` does in Session 2. If accuracy holds up on that harder set, the shortcut hypothesis is dead. **Distinguishing "learned the task" from "learned the dataset" is the first question to ask of any accuracy number**, and it is especially pressing when the number is 100%.

**Note the architecture choices that make interpretability feasible here.** Two blocks, $d = 32$, four heads, no dropout, and `norm_first` — small enough that every hidden state can be probed exhaustively and every layer patched. **Real interpretability work fights scale constantly**; this model is deliberately in the regime where exhaustive analysis is cheap.

**One structural detail that will matter in Session 2.** The head reads `h2.mean(1)` — a **mean-pool over all 16 positions**. So the verdict is an average of per-position contributions, and no single position can carry more than about $1/16$ of it by construction. That fact explains the patching numbers before you see them, and it is an architectural property rather than a discovery about the mechanism.

**Finally, keep the epistemic order straight.** The accuracy establishes that a sufficient computation exists inside this network. It says nothing about what that computation is, whether it resembles the human algorithm, or whether it generalises past the training distribution. **Everything after this cell is the work of turning "it works" into "here is how".**

In [ ]:
# probe every layer for the RUNNING DEPTH (per position) — where does the algorithm live?

# YOUR CODE HERE


**What just happened.** A linear probe for the running depth, layer by layer:

| layer | what it can see | $R^2$ |
|---|---|---|
| 0 — embeddings | current token + position only | **0.120** |
| 1 — after block 1 | + one round of attention | 0.431 |
| 2 — after block 2 | + two rounds | **0.496** |

**The rise from 0.12 to 0.50 is the finding, and its mechanism is attention.** An embedding knows one token; the running depth is a **cumulative** quantity requiring information from every earlier position. Attention is the only operation in the architecture that moves information across positions, so depth-like structure could not appear before the blocks and does appear after them. **The layer index is doing causal work here**, which is what makes the trend more informative than any single number.

**But the layer-0 value is the one to stop on, because it should be zero and is not.** Embeddings genuinely cannot know the running depth — they see one token and one position. So where does $R^2 = 0.12$ come from? **Position is correlated with depth**: early positions have small depths, later ones larger, purely as a statistical fact about the string distribution. The probe is reading a **confound**, not a representation.

**That is the central methodological warning of the session, and it generalises far beyond this demo.** A probe measures whether a quantity is **decodable**, and decodability can come from correlation, from a confound, or from the probe itself doing the computation. Probe a large enough hidden state with a flexible enough probe and you can decode almost anything — including quantities the model demonstrably does not use. **Finding a probe that works is evidence a representation *might* be there. It is not evidence the model uses it.**

**And be careful with the 0.496 too, since the printed conclusion overstates it.** Half the variance of the running depth is *linearly* recoverable from layer 2. The honest claim is exactly that — not "the model represents depth". The other half may be encoded nonlinearly, or the model may be tracking a different sufficient statistic that happens to correlate with depth at $R^2 = 0.5$. **"Depth is represented" and "half of depth's variance is linearly decodable" are different sentences**, and only the second is supported by this cell.

**A cheaper, sharper version of this experiment is worth suggesting.** Probe *per position* rather than pooling all $3000 \times 16$ states together. Depth is a per-position quantity, and pooling hides whether the representation is uniform across the sequence or concentrated near the end where the verdict is decided. A per-position $R^2$ curve is a much better picture of the mechanism and costs one extra loop.

**Finally, note exactly what question remains open.** This is the associational half of the investigation — the regression, in the language of [Causal Inference](./Causal_Inference.ipynb). Whether the model *uses* what the probe found is an **interventional** question, and no probe of any sophistication answers it. **Session 2 is the do-operator**, and it is the reason this workshop does not stop here.

---
### 🕐 Session 2 of 2 — *Activation Patching: the Causal Test* (~40 min)
**Goal:** swap internal activations between clean and corrupted runs — which components MATTER?
**Builds on:** Session 1; [Causal Inference](./Causal_Inference.ipynb).

---

## 3. From Correlation to Mechanism

💡 **Intuition.** A probe finding depth proves the information is *present*, not that it's *used* — the [collider lesson](./Causal_Inference.ipynb) for neural nets. **Activation patching** is the intervention: run a balanced string and an unbalanced one; copy one layer's activations from the balanced run into the unbalanced run; if the verdict flips toward 'balanced', that layer *causally carries* the verdict. Do it per layer and position and you map the circuit. This do-operator-for-networks is the core method of modern interpretability research.

In [ ]:
# clean = balanced string; corrupted = same string with ONE paren flipped
# single-position patches are diluted by the 16-position mean-pool — patch WHOLE layers too

# YOUR CODE HERE


**What just happened.** Two very different measurements, and only one of them means anything:

| patch | block 1 | block 2 |
|---|---|---|
| whole layer | **+1.00** | **+1.00** |
| mean single position | +0.05 | +0.06 |

**Start with the +1.00, because it is close to tautological and should not be read as a finding.** Patching *all* of block 1's output replaces the **entire residual stream**; block 2 then recomputes from clean input and the head sees a clean representation, so of course the clean verdict comes out. **A full-layer patch recovers ~1.00 at any layer of any network** — it confirms the hooks are wired correctly and nothing more. It is a harness test, not evidence that either block is where the verdict lives.

**The informative number is the single-position mean, and it says the mechanism is *distributed*.** About 0.05 per position, sixteen positions, summing to roughly 0.8 — close to full recovery spread nearly evenly. **No individual position carries the verdict.** Given that the head is `h2.mean(1)`, a mean-pool over all 16 positions, that is very close to what the architecture forces: each position can contribute at most about $1/16 = 0.0625$ of the pooled output.

**So the printed conclusion — "a localized circuit, causally mapped" — is stronger than the data supports.** The measured pattern is "distributed, and about as diluted as the mean-pool predicts". **A uniform result is a null result**, and calling it localisation reads structure into a flat line. The genuinely interesting question is whether any position *exceeds* the 0.05 baseline, which is what the heatmap is for and what the summary numbers average away.

**And there is a design flaw making the heatmap harder to read than it needs to be.** The flip position is **random for every pair** (`rng.integers(2, L_seq-2)`), but recovery is averaged over **absolute** position. So whatever structure exists around the corruption site is smeared across all positions by the averaging. **Align to the flip instead** — record recovery at flip$-2$, flip$-1$, flip, flip$+1$, … and average those. That single change is the difference between a flat heatmap and a readable one, and it is about fifteen lines.

**What the cell does establish, and it is worth stating clearly, is the method.** Run a clean input and a corrupted one differing by **exactly one character**; transplant an internal activation from clean into corrupted; measure how far the verdict moves, normalised by the clean/corrupted gap. **That is Pearl's $do(\cdot)$ implemented with a forward hook** — an intervention, not an observation, and therefore capable of answering the question a probe cannot.

**Which closes the loop the workshop opened.** Session 1's probe found depth-like structure at $R^2 = 0.50$ and could not say whether the model uses it — the same limitation the [Causal Inference](./Causal_Inference.ipynb) workshop identified for regression on observational data. Patching is the intervention. **The result here is that the verdict is carried diffusely rather than by an identifiable component**, which is a real answer, just not a tidy one.

**Finally, the honest note about the field.** Activation patching is how induction heads and the indirect-object-identification circuit were found in real models — it works. It is also expensive (one forward pass per patch site), sensitive to how you choose the corruption, and silent about whether a recovered circuit is the only one. **Scaling it is an open problem**, which is why this is an actively hiring research area rather than a solved technique.

## 4. Conclusion

Probes locate representations (depth R² rising through the blocks); patching tests *use* (verdict recovery mapped by layer and position). Correlation-to-causation, inside the network — the same discipline [Causal Inference](./Causal_Inference.ipynb) taught for the world outside. Scaling these methods to frontier models is an open, hiring-hot research field.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — apply both tools to the fable nano-GPT you trained there.
- [Causal Inference](./Causal_Inference.ipynb) — the intervention logic, formalized.